# Module 2: Data Cleaning and Structuring

This notebook cleans the raw military dataset and prepares it for Tableau.

### Steps Performed
1. Load the raw dataset
2. Clean and standardize the data
3. Convert columns to numeric format
4. Handle missing values
5. Validate the dataset
6. Save the cleaned dataset as `military_cleaned.csv`


## Step 1: Imports and file paths

In [1]:
import os
import re
import pandas as pd

# This notebook lives in scripts/, so the data folder is one level up.
RAW_CSV_PATH = os.path.join("data", "military_raw_data.csv")
CLEANED_CSV_PATH = os.path.join("data", "military_cleaned.csv")

print("Reading from:", os.path.abspath(RAW_CSV_PATH))
print("Will save to:", os.path.abspath(CLEANED_CSV_PATH))


Reading from: /Users/mac/Desktop/Unified-Military-Analytics/scripts/data/military_raw_data.csv
Will save to: /Users/mac/Desktop/Unified-Military-Analytics/scripts/data/military_cleaned.csv


## Step 2: Load the raw dataset

This is the exact output of Module 1 -- nothing has been touched yet.

In [2]:
df = pd.read_csv(RAW_CSV_PATH)

print("RAW dataset loaded.")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()


RAW dataset loaded.
Rows: 145, Columns: 57


,country,power_index_rank,power_index_score,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,...,proven_oil_reserves_bbl,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km
0,United States,1,0.0741,3.419634e+08,150463900.0,124816644.0,4445524.0,1333030.0,799500.0,0.0,...,3.821200e+10,1.029000e+12,9.143010e+11,1.340200e+13,5.342340e+08,4.951560e+08,2.478830e+11,9833517.0,19924.0,12002.0
1,Russia,2,0.0791,1.408208e+08,69002197.0,46189226.0,1267387.0,1320000.0,2000000.0,250000.0,...,8.000000e+10,6.178300e+11,4.722390e+11,4.780500e+13,5.311300e+08,2.907630e+08,1.621660e+11,17098242.0,37653.0,22407.0
2,China,3,0.0919,1.415043e+09,764123366.0,626864169.0,19810606.0,2035000.0,510000.0,625000.0,...,2.602300e+10,2.253410e+11,3.661600e+11,6.654000e+12,4.805000e+09,5.191000e+09,1.570410e+11,9596960.0,14500.0,22457.0
3,India,4,0.1346,1.409128e+09,662290299.0,522786598.0,23955181.0,1431000.0,1000000.0,2527000.0,...,4.605000e+09,3.317000e+10,5.886700e+10,1.381000e+12,1.020000e+09,1.262000e+09,1.277270e+11,3287263.0,7000.0,13888.0
4,South Korea,5,0.1642,5.208180e+07,26040900.0,21353538.0,416654.0,450000.0,3100000.0,120000.0,...,0.000000e+00,5.512700e+07,5.948000e+10,7.079000e+09,1.608100e+07,1.368170e+08,3.260000e+08,99720.0,2413.0,237.0


## Step 3: Standardize column names

Even though the scraper already used clean snake_case names, we run this defensively --
if a column ever had a space, a hyphen, or a stray symbol, this guarantees every column
name is lowercase, uses underscores, and has no special characters. This is what your
mentor means by "standardize column names".

In [3]:
def standardize_column_name(name):
    name = name.strip().lower()
    name = re.sub(r"[\s\-]+", "_", name)       # spaces/hyphens -> underscore
    name = re.sub(r"[^a-z0-9_]", "", name)      # drop anything not a-z, 0-9, or _
    name = re.sub(r"_+", "_", name).strip("_")  # collapse repeated underscores
    return name

original_columns = list(df.columns)
df.columns = [standardize_column_name(c) for c in df.columns]

renamed = {old: new for old, new in zip(original_columns, df.columns) if old != new}
if renamed:
    print(f"Renamed {len(renamed)} column(s):")
    for old, new in renamed.items():
        print(f"  '{old}' -> '{new}'")
else:
    print("All column names were already clean -- no renaming needed.")


All column names were already clean -- no renaming needed.


## Step 4: Clean the `country` column

- Remove leading/trailing whitespace (e.g. `"France "` -> `"France"`)
- Remove exact duplicate country rows, keeping the first occurrence

In [4]:
df["country"] = df["country"].astype(str).str.strip()

rows_before = len(df)
df = df.drop_duplicates(subset="country", keep="first").reset_index(drop=True)
rows_after = len(df)

print(f"Duplicate rows removed: {rows_before - rows_after}")
print(f"Rows remaining: {rows_after}")


Duplicate rows removed: 0
Rows remaining: 145


## Step 5: Replace invalid placeholder text with real missing values

GlobalFirepower (and messy data in general) sometimes uses placeholder text instead of
leaving a cell truly empty -- things like `-`, `--`, `N/A`, `NA`, or just spaces.
Pandas doesn't know these mean "missing" unless we tell it. We replace all of them
with a real `NaN` so every later calculation treats them consistently.

In [5]:
INVALID_PLACEHOLDERS = ["-", "--", "N/A", "NA", "n/a", "na", "None", "none", "null", ""]

metric_columns = [c for c in df.columns if c != "country"]

for column in metric_columns:
    # Only touch columns that are still text (object dtype) -- numeric columns
    # can't contain these placeholder strings anymore.
    if df[column].dtype == object:
        df[column] = df[column].astype(str).str.strip()
        df[column] = df[column].replace(INVALID_PLACEHOLDERS, pd.NA)

print("Invalid placeholder values replaced with NaN across all metric columns.")


Invalid placeholder values replaced with NaN across all metric columns.


## Step 6: Convert every metric column to a proper numeric type

Even though the scraper already stored most values as numbers, this step is a safety
net: it strips any leftover commas, `%`, or `+` symbols, then forces every metric
column into a real numeric dtype using `pd.to_numeric(..., errors="coerce")`.
Anything that still can't be converted becomes `NaN` instead of crashing the notebook.

In [6]:
def clean_and_convert_to_numeric(series):
    # Remove common junk characters that sometimes appear in scraped text data.
    cleaned = series.astype(str).str.replace(",", "", regex=False)
    cleaned = cleaned.str.replace("%", "", regex=False)
    cleaned = cleaned.str.replace("+", "", regex=False)
    cleaned = cleaned.str.strip()
    return pd.to_numeric(cleaned, errors="coerce")

for column in metric_columns:
    df[column] = clean_and_convert_to_numeric(df[column])

print("All metric columns converted to numeric.")
print(df.dtypes)


All metric columns converted to numeric.
country                                          str
power_index_rank                               int64
power_index_score                            float64
total_population                             float64
total_military_manpower                      float64
fit_for_service                              float64
population_reaching_military_age_annually    float64
active_personnel                             float64
reserve_personnel                            float64
paramilitary                                 float64
total_military_aircraft                      float64
fighter_aircraft                             float64
attack_aircraft                              float64
transport_aircraft                           float64
trainer_aircraft                             float64
special_mission_aircraft                     float64
tanker_aircraft                              float64
total_military_helicopters                   float64
attac

## Step 7: Missing value report -- BEFORE intelligent filling

In [7]:
def missing_value_report(dataframe, label):
    total_cells = dataframe.size
    missing_cells = dataframe.isna().sum().sum()
    missing_pct = (missing_cells / total_cells) * 100

    print(f"--- Missing value report: {label} ---")
    print(f"Rows: {dataframe.shape[0]}, Columns: {dataframe.shape[1]}")
    print(f"Missing cells: {missing_cells} / {total_cells} ({missing_pct:.2f}%)")

    per_column = dataframe.isna().sum().sort_values(ascending=False)
    per_column = per_column[per_column > 0]
    if len(per_column) > 0:
        print("Top columns with missing values:")
        for col, count in per_column.head(15).items():
            print(f"  - {col:<40} {count:>4} missing")
    else:
        print("No missing values.")
    print()
    return missing_pct

missing_pct_before = missing_value_report(df, "BEFORE intelligent filling")


--- Missing value report: BEFORE intelligent filling ---
Rows: 145, Columns: 57
Missing cells: 94 / 8265 (1.14%)
Top columns with missing values:
  - total_naval_fleet_tonnage_mt               94 missing



## Step 8: Intelligently fill missing military hardware/asset counts with 0

This is the key judgment call in Module 2. For equipment/hardware COUNT columns
(tanks, submarines, aircraft carriers, etc.), GlobalFirepower simply doesn't list a
country on that page if it owns **zero** of that item -- it does NOT mean the data is
unknown. So a missing value here should become `0`, a real and meaningful fact,
not stay as `NaN`.

This is different from economic/geographic columns (population, GDP-related figures,
land area, etc.), where a missing value is more likely a genuine data gap -- so we
leave those as `NaN` rather than guessing a number.

In [8]:
# Columns where "missing" really means "owns zero of this" -- safe to fill with 0.
ASSET_COUNT_COLUMNS = [
    # Air Force
    "total_military_aircraft", "fighter_aircraft", "attack_aircraft",
    "transport_aircraft", "trainer_aircraft", "special_mission_aircraft",
    "tanker_aircraft", "total_military_helicopters", "attack_helicopters",
    # Land Forces
    "tanks", "armored_fighting_vehicles", "self_propelled_artillery",
    "towed_artillery", "rocket_projectors",
    # Navy
    "total_naval_fleet", "total_naval_fleet_tonnage_mt", "aircraft_carriers",
    "helicopter_carriers", "submarines", "destroyers", "frigates",
    "corvettes", "coastal_patrol_craft", "mine_warfare_craft",
]

filled_summary = {}
for column in ASSET_COUNT_COLUMNS:
    if column in df.columns:
        missing_count = df[column].isna().sum()
        if missing_count > 0:
            df[column] = df[column].fillna(0)
            filled_summary[column] = missing_count

print(f"Filled {sum(filled_summary.values())} missing values across {len(filled_summary)} hardware columns:")
for col, count in filled_summary.items():
    print(f"  - {col:<40} {count:>4} values set to 0")


Filled 94 missing values across 1 hardware columns:
  - total_naval_fleet_tonnage_mt               94 values set to 0


## Step 9: Missing value report -- AFTER intelligent filling

In [9]:
missing_pct_after = missing_value_report(df, "AFTER intelligent filling")

print(f"Missing value change: {missing_pct_before:.2f}% -> {missing_pct_after:.2f}%")

if missing_pct_after < 2:
    print("PASS: under the 2% missing-data target.")
else:
    print("WARNING: still above the 2% missing-data target -- see remaining columns above.")


--- Missing value report: AFTER intelligent filling ---
Rows: 145, Columns: 57
Missing cells: 0 / 8265 (0.00%)
No missing values.

Missing value change: 1.14% -> 0.00%
PASS: under the 2% missing-data target.


## Step 10: Structural integrity checks

Final sanity checks before saving, matching your mentor's evaluation criteria:
"No structural errors in the output dataset"

In [10]:
print("Final row count:", df.shape[0])
print("Final column count:", df.shape[1])
print("Duplicate country names:", df["country"].duplicated().sum())
print("Blank country names:", (df["country"].str.strip() == "").sum())
print("Any fully-empty columns:", (df.isna().all()).sum())
print()
print("Column data types:")
print(df.dtypes)


Final row count: 145
Final column count: 57
Duplicate country names: 0
Blank country names: 0
Any fully-empty columns: 0

Column data types:
country                                          str
power_index_rank                               int64
power_index_score                            float64
total_population                             float64
total_military_manpower                      float64
fit_for_service                              float64
population_reaching_military_age_annually    float64
active_personnel                             float64
reserve_personnel                            float64
paramilitary                                 float64
total_military_aircraft                      float64
fighter_aircraft                             float64
attack_aircraft                              float64
transport_aircraft                           float64
trainer_aircraft                             float64
special_mission_aircraft                     float64
tanker_airc

## Step 11: Save the cleaned dataset

In [11]:
os.makedirs(os.path.dirname(CLEANED_CSV_PATH), exist_ok=True)
df.to_csv(CLEANED_CSV_PATH, index=False)

print(f"Cleaned dataset saved -> {os.path.abspath(CLEANED_CSV_PATH)}")
print(f"Final shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(10)


Cleaned dataset saved -> /Users/mac/Desktop/Unified-Military-Analytics/scripts/data/military_cleaned.csv
Final shape: 145 rows x 57 columns


,country,power_index_rank,power_index_score,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,...,proven_oil_reserves_bbl,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km
0,United States,1,0.0741,3.419634e+08,150463900.0,124816644.0,4445524.0,1333030.0,799500.0,0.0,...,3.821200e+10,1.029000e+12,9.143010e+11,1.340200e+13,5.342340e+08,4.951560e+08,2.478830e+11,9833517.0,19924.0,12002.0
1,Russia,2,0.0791,1.408208e+08,69002197.0,46189226.0,1267387.0,1320000.0,2000000.0,250000.0,...,8.000000e+10,6.178300e+11,4.722390e+11,4.780500e+13,5.311300e+08,2.907630e+08,1.621660e+11,17098242.0,37653.0,22407.0
2,China,3,0.0919,1.415043e+09,764123366.0,626864169.0,19810606.0,2035000.0,510000.0,625000.0,...,2.602300e+10,2.253410e+11,3.661600e+11,6.654000e+12,4.805000e+09,5.191000e+09,1.570410e+11,9596960.0,14500.0,22457.0
3,India,4,0.1346,1.409128e+09,662290299.0,522786598.0,23955181.0,1431000.0,1000000.0,2527000.0,...,4.605000e+09,3.317000e+10,5.886700e+10,1.381000e+12,1.020000e+09,1.262000e+09,1.277270e+11,3287263.0,7000.0,13888.0
4,South Korea,5,0.1642,5.208180e+07,26040900.0,21353538.0,416654.0,450000.0,3100000.0,120000.0,...,0.000000e+00,5.512700e+07,5.948000e+10,7.079000e+09,1.608100e+07,1.368170e+08,3.260000e+08,99720.0,2413.0,237.0
5,France,6,0.1798,6.837459e+07,30084820.0,23794358.0,752121.0,264000.0,43444.0,150000.0,...,6.171900e+07,2.013200e+07,3.700100e+10,7.787000e+09,2.157000e+06,1.257000e+07,1.600000e+08,643801.0,4853.0,4072.0
6,Japan,7,0.1876,1.232019e+08,52976836.0,42874277.0,1108818.0,251500.0,56000.0,25000.0,...,4.411500e+07,2.220000e+09,9.284300e+10,2.089800e+10,2.765700e+07,1.976120e+08,3.500000e+08,377915.0,29751.0,0.0
7,United Kingdom,8,0.1881,6.845906e+07,31491165.0,25192932.0,753050.0,141330.0,31940.0,0.0,...,2.500000e+09,3.775800e+10,7.014100e+10,1.806610e+11,1.568000e+06,7.372000e+06,2.600000e+07,243610.0,12429.0,499.0
8,Turkiye,9,0.1975,8.411953e+07,42985080.0,36087279.0,1430032.0,481000.0,380000.0,150000.0,...,3.660000e+08,3.797010e+08,5.288700e+10,3.794000e+09,8.253400e+07,1.241830e+08,1.097500e+10,783562.0,7200.0,2816.0
9,Italy,10,0.2211,6.096493e+07,27434219.0,22191235.0,548684.0,165500.0,18300.0,105000.0,...,4.979340e+08,3.117000e+09,6.873500e+10,4.575000e+10,1.572000e+06,1.242400e+07,6.099990e+08,301340.0,7600.0,1836.0
